In [1]:
import os
import pickle
import random
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [2]:
import sys, types
import numpy.core.numeric as _nc
_np_core = types.ModuleType("numpy._core")
_np_core.numeric = types.ModuleType("numpy._core.numeric")
_np_core.numeric.__dict__.update(_nc.__dict__)
sys.modules["numpy._core"] = _np_core
sys.modules["numpy._core.numeric"] = _np_core.numeric

# SCP Method - Coverage and Interval Size

In [3]:
base_path      = r"C:\Users\MGA5500\Desktop\PROJECT\WW\Data"
all_eval_seeds = list(range(60, 70))
nx             = 250
alpha          = 0.10
slices         = {'u': slice(0, nx), 'h': slice(nx, 2*nx), 'r': slice(2*nx, 3*nx)}
I              = 10    
T_start        = 20
T              = 201   

In [4]:
def load_ensemble(seed, t):
    path = os.path.join(base_path, str(seed))
    with open(os.path.join(path, 'NN_data.pkl'), 'rb') as f:
        nn = pickle.load(f)
    with open(os.path.join(path, 'QPEns_data.pkl'), 'rb') as f:
        qpens = pickle.load(f)
    nn_arr    = np.asarray(nn['analysis'][t])      
    qpens_arr = np.asarray(qpens['analysis'][t])
    nn_ens    = {v: nn_arr[sl, :]    for v, sl in slices.items()}
    qpens_ens = {v: qpens_arr[sl, :] for v, sl in slices.items()}
    return nn_ens, qpens_ens


coverage_all      = {v: np.zeros((I, T)) for v in slices}
interval_size_all = {v: np.zeros((I, T)) for v in slices}

for i in range(I):
    print(f"\n=== Iteration {i+1}/{I} ===")
    seeds = all_eval_seeds.copy()
    random.shuffle(seeds)
    calib_seeds = seeds[:len(seeds)//2]
    test_seeds  = seeds[len(seeds)//2:]
    print(f"calib={calib_seeds}, test={test_seeds}")

    for t in range(T_start, T): 
        # calibration
        nn_cal, qp_cal = {v: [] for v in slices}, {v: [] for v in slices}
        for seed in calib_seeds:
            try:
                nn_ens, qpens_ens = load_ensemble(seed, t)
                for v in slices:
                    nn_cal[v].append(nn_ens[v])
                    qp_cal[v].append(qpens_ens[v])
            except Exception as e:
                print(f"calib seed={seed}, t={t}: {e}")

        thresh = {}
        for v in slices:
            if not nn_cal[v]:
                thresh[v] = np.nan
                continue
            nn_stack = np.stack(nn_cal[v], axis=0)
            qp_stack = np.stack(qp_cal[v], axis=0)
            scores = np.abs(qp_stack - nn_stack).ravel()
            N      = scores.size
            qprob  = float(np.ceil((N + 1) * (1 - alpha)) / N)
            thresh[v] = float(np.quantile(scores, qprob, method="inverted_cdf"))

        # test
        nn_test, qp_test = {v: [] for v in slices}, {v: [] for v in slices}
        for seed in test_seeds:
            try:
                nn_ens, qpens_ens = load_ensemble(seed, t)
                for v in slices:
                    nn_test[v].append(nn_ens[v])
                    qp_test[v].append(qpens_ens[v])
            except Exception as e:
                print(f"test seed={seed}, t={t}: {e}")

        for v in slices:
            qv = thresh[v]
            if not np.isfinite(qv) or not nn_test[v]:
                coverage_all[v][i, t] = np.nan
                interval_size_all[v][i, t] = np.nan
                continue
            nn_stack = np.stack(nn_test[v], axis=0)
            qp_stack = np.stack(qp_test[v], axis=0)
            lower = np.maximum(nn_stack - qv, 0)
            upper = np.maximum(nn_stack + qv, 0)
            in_int = (qp_stack >= lower) & (qp_stack <= upper)
            coverage_all[v][i, t]      = 100.0 * float(np.mean(in_int))
            interval_size_all[v][i, t] = float(np.mean(upper - lower))

        print(f"  t={t}: " + "  ".join(
            f"{v} q={thresh[v]:.4f} cov={coverage_all[v][i,t]:.1f}%"
            for v in slices))
        
export = {'time_step': np.arange(T_start, T)}
for v in slices:
    export[f'{v}_avg_coverage']      = np.nanmean(coverage_all[v],      axis=0)[T_start:]
    export[f'{v}_std_coverage']      = np.nanstd(coverage_all[v],       axis=0)[T_start:]
    export[f'{v}_avg_interval_size'] = np.nanmean(interval_size_all[v], axis=0)[T_start:]
    export[f'{v}_std_interval_size'] = np.nanstd(interval_size_all[v],  axis=0)[T_start:]

df = pd.DataFrame(export).set_index('time_step')
csv_path = "scp_coverage_interval.csv"
df.to_csv(csv_path)
print(f"Saved: {csv_path}")
print(df.head())


=== Iteration 1/10 ===
calib=[60, 65, 61, 63, 69], test=[68, 67, 66, 62, 64]
  t=20: u q=0.0034 cov=89.9%  h q=0.0672 cov=88.5%  r q=0.0056 cov=88.5%
  t=21: u q=0.0032 cov=88.8%  h q=0.0526 cov=88.4%  r q=0.0045 cov=88.6%
  t=22: u q=0.0029 cov=89.4%  h q=0.0348 cov=88.5%  r q=0.0030 cov=88.9%
  t=23: u q=0.0028 cov=90.0%  h q=0.0269 cov=89.1%  r q=0.0022 cov=88.7%
  t=24: u q=0.0026 cov=89.8%  h q=0.0203 cov=88.7%  r q=0.0016 cov=87.9%
  t=25: u q=0.0026 cov=89.2%  h q=0.0209 cov=89.8%  r q=0.0013 cov=88.2%
  t=26: u q=0.0026 cov=90.3%  h q=0.0182 cov=88.9%  r q=0.0016 cov=89.0%
  t=27: u q=0.0023 cov=87.7%  h q=0.0176 cov=88.8%  r q=0.0014 cov=89.3%
  t=28: u q=0.0023 cov=86.1%  h q=0.0138 cov=85.7%  r q=0.0010 cov=87.0%
  t=29: u q=0.0023 cov=88.0%  h q=0.0151 cov=87.3%  r q=0.0011 cov=87.9%
  t=30: u q=0.0022 cov=88.9%  h q=0.0165 cov=87.9%  r q=0.0012 cov=87.7%
  t=31: u q=0.0022 cov=87.7%  h q=0.0152 cov=87.8%  r q=0.0011 cov=88.5%
  t=32: u q=0.0023 cov=89.1%  h q=0.0166 cov=8

  t=131: u q=0.0021 cov=88.6%  h q=0.0116 cov=87.6%  r q=0.0006 cov=86.0%
  t=132: u q=0.0020 cov=88.5%  h q=0.0106 cov=86.1%  r q=0.0006 cov=85.9%
  t=133: u q=0.0020 cov=89.2%  h q=0.0110 cov=87.5%  r q=0.0006 cov=84.9%
  t=134: u q=0.0022 cov=89.9%  h q=0.0140 cov=89.4%  r q=0.0008 cov=88.0%
  t=135: u q=0.0021 cov=89.9%  h q=0.0133 cov=90.5%  r q=0.0009 cov=88.1%
  t=136: u q=0.0020 cov=91.0%  h q=0.0120 cov=90.4%  r q=0.0008 cov=90.5%
  t=137: u q=0.0022 cov=91.2%  h q=0.0125 cov=89.7%  r q=0.0006 cov=89.2%
  t=138: u q=0.0020 cov=89.3%  h q=0.0119 cov=88.5%  r q=0.0007 cov=88.3%
  t=139: u q=0.0020 cov=88.2%  h q=0.0124 cov=88.1%  r q=0.0008 cov=88.4%
  t=140: u q=0.0021 cov=88.9%  h q=0.0145 cov=88.7%  r q=0.0008 cov=86.5%
  t=141: u q=0.0022 cov=89.3%  h q=0.0151 cov=88.2%  r q=0.0012 cov=88.7%
  t=142: u q=0.0021 cov=88.6%  h q=0.0153 cov=89.9%  r q=0.0010 cov=89.0%
  t=143: u q=0.0020 cov=88.3%  h q=0.0136 cov=88.7%  r q=0.0010 cov=88.5%
  t=144: u q=0.0021 cov=87.8%  h q=0.0

  t=61: u q=0.0022 cov=89.2%  h q=0.0169 cov=89.6%  r q=0.0012 cov=89.1%
  t=62: u q=0.0022 cov=88.3%  h q=0.0133 cov=87.1%  r q=0.0009 cov=88.0%
  t=63: u q=0.0022 cov=88.5%  h q=0.0161 cov=87.5%  r q=0.0013 cov=88.4%
  t=64: u q=0.0022 cov=89.3%  h q=0.0142 cov=87.5%  r q=0.0011 cov=88.4%
  t=65: u q=0.0021 cov=87.2%  h q=0.0164 cov=89.6%  r q=0.0011 cov=88.8%
  t=66: u q=0.0022 cov=90.2%  h q=0.0172 cov=90.8%  r q=0.0015 cov=91.1%
  t=67: u q=0.0023 cov=89.4%  h q=0.0162 cov=89.1%  r q=0.0013 cov=89.9%
  t=68: u q=0.0024 cov=90.0%  h q=0.0187 cov=90.4%  r q=0.0015 cov=90.1%
  t=69: u q=0.0024 cov=91.8%  h q=0.0179 cov=91.3%  r q=0.0018 cov=91.4%
  t=70: u q=0.0023 cov=90.7%  h q=0.0157 cov=89.6%  r q=0.0015 cov=89.8%
  t=71: u q=0.0022 cov=88.9%  h q=0.0166 cov=89.6%  r q=0.0016 cov=90.7%
  t=72: u q=0.0023 cov=89.9%  h q=0.0162 cov=89.1%  r q=0.0016 cov=90.5%
  t=73: u q=0.0024 cov=89.9%  h q=0.0175 cov=89.7%  r q=0.0016 cov=89.9%
  t=74: u q=0.0024 cov=91.2%  h q=0.0175 cov=92.7% 

  t=173: u q=0.0022 cov=89.5%  h q=0.0141 cov=89.2%  r q=0.0011 cov=89.9%
  t=174: u q=0.0023 cov=90.6%  h q=0.0158 cov=90.8%  r q=0.0010 cov=90.3%
  t=175: u q=0.0025 cov=91.5%  h q=0.0174 cov=90.7%  r q=0.0013 cov=90.4%
  t=176: u q=0.0025 cov=92.0%  h q=0.0208 cov=92.6%  r q=0.0017 cov=91.1%
  t=177: u q=0.0024 cov=91.7%  h q=0.0190 cov=92.5%  r q=0.0017 cov=93.2%
  t=178: u q=0.0025 cov=92.8%  h q=0.0169 cov=92.3%  r q=0.0012 cov=91.8%
  t=179: u q=0.0023 cov=91.3%  h q=0.0146 cov=91.8%  r q=0.0010 cov=91.9%
  t=180: u q=0.0023 cov=91.7%  h q=0.0134 cov=91.4%  r q=0.0009 cov=91.3%
  t=181: u q=0.0022 cov=90.4%  h q=0.0128 cov=90.4%  r q=0.0008 cov=90.8%
  t=182: u q=0.0023 cov=91.0%  h q=0.0153 cov=91.5%  r q=0.0009 cov=91.0%
  t=183: u q=0.0025 cov=92.1%  h q=0.0173 cov=91.3%  r q=0.0010 cov=91.2%
  t=184: u q=0.0024 cov=91.3%  h q=0.0202 cov=92.2%  r q=0.0018 cov=93.8%
  t=185: u q=0.0024 cov=90.9%  h q=0.0184 cov=89.4%  r q=0.0013 cov=90.4%
  t=186: u q=0.0024 cov=89.6%  h q=0.0

  t=103: u q=0.0024 cov=92.9%  h q=0.0208 cov=92.5%  r q=0.0016 cov=92.3%
  t=104: u q=0.0025 cov=92.0%  h q=0.0191 cov=91.2%  r q=0.0016 cov=91.2%
  t=105: u q=0.0024 cov=90.2%  h q=0.0188 cov=89.8%  r q=0.0014 cov=89.6%
  t=106: u q=0.0023 cov=89.6%  h q=0.0176 cov=89.0%  r q=0.0014 cov=88.3%
  t=107: u q=0.0023 cov=88.9%  h q=0.0158 cov=88.9%  r q=0.0012 cov=88.7%
  t=108: u q=0.0023 cov=89.0%  h q=0.0174 cov=90.3%  r q=0.0015 cov=89.7%
  t=109: u q=0.0022 cov=88.4%  h q=0.0153 cov=90.3%  r q=0.0015 cov=92.0%
  t=110: u q=0.0021 cov=88.6%  h q=0.0127 cov=89.8%  r q=0.0008 cov=89.4%
  t=111: u q=0.0021 cov=89.0%  h q=0.0123 cov=89.5%  r q=0.0008 cov=88.8%
  t=112: u q=0.0020 cov=89.7%  h q=0.0132 cov=90.2%  r q=0.0009 cov=89.5%
  t=113: u q=0.0019 cov=87.5%  h q=0.0107 cov=86.0%  r q=0.0007 cov=87.2%
  t=114: u q=0.0021 cov=86.9%  h q=0.0119 cov=85.1%  r q=0.0008 cov=86.9%
  t=115: u q=0.0020 cov=87.2%  h q=0.0126 cov=86.3%  r q=0.0006 cov=83.9%
  t=116: u q=0.0021 cov=88.2%  h q=0.0

...

  t=87: u q=0.0021 cov=86.9%  h q=0.0152 cov=87.8%  r q=0.0013 cov=89.5%
  t=88: u q=0.0023 cov=88.7%  h q=0.0177 cov=88.4%  r q=0.0016 cov=89.1%
  t=89: u q=0.0024 cov=89.3%  h q=0.0182 cov=88.8%  r q=0.0017 cov=89.6%
  t=90: u q=0.0022 cov=87.9%  h q=0.0187 cov=88.7%  r q=0.0016 cov=88.5%
  t=91: u q=0.0024 cov=90.0%  h q=0.0179 cov=90.0%  r q=0.0017 cov=90.6%
  t=92: u q=0.0024 cov=90.7%  h q=0.0182 cov=90.8%  r q=0.0018 cov=91.8%
  t=93: u q=0.0023 cov=91.0%  h q=0.0166 cov=90.8%  r q=0.0015 cov=91.2%
  t=94: u q=0.0024 cov=90.9%  h q=0.0186 cov=91.2%  r q=0.0018 cov=91.9%
  t=95: u q=0.0022 cov=90.6%  h q=0.0166 cov=91.0%  r q=0.0012 cov=91.4%
  t=96: u q=0.0022 cov=90.5%  h q=0.0180 cov=91.1%  r q=0.0013 cov=91.0%
  t=97: u q=0.0022 cov=89.8%  h q=0.0131 cov=88.8%  r q=0.0011 cov=89.9%
  t=98: u q=0.0022 cov=89.4%  h q=0.0158 cov=90.5%  r q=0.0011 cov=89.3%
  t=99: u q=0.0023 cov=90.2%  h q=0.0161 cov=91.5%  r q=0.0011 cov=90.2%
  t=100: u q=0.0021 cov=89.6%  h q=0.0141 cov=90.3%

  t=198: u q=0.0025 cov=91.7%  h q=0.0189 cov=91.1%  r q=0.0018 cov=90.8%
  t=199: u q=0.0024 cov=92.0%  h q=0.0161 cov=90.6%  r q=0.0011 cov=88.6%
  t=200: u q=0.0022 cov=89.1%  h q=0.0135 cov=87.2%  r q=0.0010 cov=87.5%

=== Iteration 9/10 ===
calib=[64, 65, 69, 60, 61], test=[66, 62, 63, 68, 67]
  t=20: u q=0.0034 cov=89.6%  h q=0.0636 cov=86.9%  r q=0.0051 cov=86.3%
  t=21: u q=0.0032 cov=88.0%  h q=0.0477 cov=86.6%  r q=0.0041 cov=87.3%
  t=22: u q=0.0028 cov=87.6%  h q=0.0329 cov=87.7%  r q=0.0028 cov=87.6%
  t=23: u q=0.0026 cov=87.6%  h q=0.0244 cov=87.7%  r q=0.0020 cov=87.5%
  t=24: u q=0.0025 cov=88.8%  h q=0.0203 cov=88.7%  r q=0.0016 cov=87.7%
  t=25: u q=0.0026 cov=89.4%  h q=0.0212 cov=90.0%  r q=0.0014 cov=89.0%
  t=26: u q=0.0027 cov=91.7%  h q=0.0201 cov=90.5%  r q=0.0019 cov=91.2%
  t=27: u q=0.0025 cov=90.2%  h q=0.0213 cov=91.7%  r q=0.0017 cov=91.3%
  t=28: u q=0.0024 cov=88.9%  h q=0.0172 cov=89.5%  r q=0.0015 cov=90.4%
  t=29: u q=0.0023 cov=89.3%  h q=0.0173 co

  t=128: u q=0.0023 cov=90.1%  h q=0.0157 cov=88.9%  r q=0.0010 cov=88.5%
  t=129: u q=0.0022 cov=88.7%  h q=0.0134 cov=88.9%  r q=0.0009 cov=88.2%
  t=130: u q=0.0020 cov=89.0%  h q=0.0098 cov=86.6%  r q=0.0006 cov=85.8%
  t=131: u q=0.0020 cov=86.8%  h q=0.0109 cov=86.4%  r q=0.0005 cov=84.7%
  t=132: u q=0.0020 cov=88.3%  h q=0.0100 cov=84.6%  r q=0.0006 cov=85.9%
  t=133: u q=0.0019 cov=88.2%  h q=0.0104 cov=86.4%  r q=0.0006 cov=86.1%
  t=134: u q=0.0021 cov=88.9%  h q=0.0133 cov=88.4%  r q=0.0007 cov=86.9%
  t=135: u q=0.0020 cov=88.9%  h q=0.0125 cov=89.3%  r q=0.0009 cov=88.6%
  t=136: u q=0.0020 cov=90.3%  h q=0.0112 cov=89.1%  r q=0.0007 cov=89.2%
  t=137: u q=0.0021 cov=89.8%  h q=0.0116 cov=88.4%  r q=0.0006 cov=89.2%
  t=138: u q=0.0020 cov=89.5%  h q=0.0136 cov=90.8%  r q=0.0009 cov=91.1%
  t=139: u q=0.0021 cov=90.3%  h q=0.0160 cov=92.4%  r q=0.0011 cov=91.5%
  t=140: u q=0.0023 cov=91.1%  h q=0.0184 cov=92.2%  r q=0.0013 cov=91.1%
  t=141: u q=0.0024 cov=91.4%  h q=0.0

  t=58: u q=0.0024 cov=91.5%  h q=0.0185 cov=90.8%  r q=0.0016 cov=91.2%
  t=59: u q=0.0026 cov=92.1%  h q=0.0229 cov=92.1%  r q=0.0018 cov=90.8%
  t=60: u q=0.0025 cov=91.8%  h q=0.0212 cov=90.8%  r q=0.0015 cov=91.3%
  t=61: u q=0.0023 cov=91.8%  h q=0.0198 cov=91.9%  r q=0.0015 cov=91.2%
  t=62: u q=0.0025 cov=92.9%  h q=0.0204 cov=93.6%  r q=0.0016 cov=93.1%
  t=63: u q=0.0025 cov=92.2%  h q=0.0219 cov=92.3%  r q=0.0020 cov=92.6%
  t=64: u q=0.0024 cov=92.6%  h q=0.0197 cov=92.9%  r q=0.0016 cov=92.0%
  t=65: u q=0.0024 cov=91.9%  h q=0.0189 cov=91.7%  r q=0.0016 cov=92.6%
  t=66: u q=0.0023 cov=90.7%  h q=0.0157 cov=89.3%  r q=0.0014 cov=90.7%
  t=67: u q=0.0024 cov=90.6%  h q=0.0182 cov=90.9%  r q=0.0015 cov=90.9%
  t=68: u q=0.0024 cov=90.8%  h q=0.0209 cov=92.0%  r q=0.0019 cov=92.4%
  t=69: u q=0.0023 cov=90.4%  h q=0.0173 cov=90.8%  r q=0.0017 cov=91.5%
  t=70: u q=0.0023 cov=90.9%  h q=0.0168 cov=90.7%  r q=0.0017 cov=91.2%
  t=71: u q=0.0023 cov=90.6%  h q=0.0173 cov=90.3% 

  t=170: u q=0.0024 cov=92.4%  h q=0.0179 cov=92.1%  r q=0.0015 cov=91.9%
  t=171: u q=0.0021 cov=89.0%  h q=0.0142 cov=89.2%  r q=0.0010 cov=89.3%
  t=172: u q=0.0021 cov=89.7%  h q=0.0126 cov=89.2%  r q=0.0009 cov=90.0%
  t=173: u q=0.0022 cov=89.2%  h q=0.0148 cov=90.2%  r q=0.0010 cov=88.9%
  t=174: u q=0.0022 cov=88.3%  h q=0.0135 cov=88.1%  r q=0.0009 cov=88.4%
  t=175: u q=0.0023 cov=89.4%  h q=0.0160 cov=89.4%  r q=0.0012 cov=89.5%
  t=176: u q=0.0023 cov=89.7%  h q=0.0165 cov=89.1%  r q=0.0014 cov=89.5%
  t=177: u q=0.0022 cov=88.8%  h q=0.0160 cov=89.6%  r q=0.0012 cov=89.6%
  t=178: u q=0.0022 cov=88.4%  h q=0.0146 cov=89.3%  r q=0.0009 cov=89.8%
  t=179: u q=0.0022 cov=89.3%  h q=0.0130 cov=89.5%  r q=0.0008 cov=90.0%
  t=180: u q=0.0021 cov=88.6%  h q=0.0105 cov=86.5%  r q=0.0007 cov=88.2%
  t=181: u q=0.0021 cov=87.1%  h q=0.0102 cov=85.1%  r q=0.0005 cov=86.6%
  t=182: u q=0.0022 cov=89.8%  h q=0.0127 cov=88.3%  r q=0.0006 cov=87.6%
  t=183: u q=0.0022 cov=88.2%  h q=0.0

# NCP Method - Coverage and Interval Size

In [21]:
base_path      = r"C:\Users\MGA5500\Desktop\PROJECT\WW\Data"
rf_train_seeds = list(range(70, 75))
all_eval_seeds = list(range(60, 70))
nx             = 250
alpha          = 0.10
slices         = {'u': slice(0, nx), 'h': slice(nx, 2*nx), 'r': slice(2*nx, 3*nx)}
I              = 10
T_start        = 20
T              = 201
exp_seed       = 1

In [22]:
def load_ensemble_ncp(seed, t):
    path = os.path.join(base_path, str(seed))
    with open(os.path.join(path, 'NN_data.pkl'), 'rb') as f:
        nn = pickle.load(f)
    with open(os.path.join(path, 'QPEns_data.pkl'), 'rb') as f:
        qpens = pickle.load(f)
    arr_nn    = np.asarray(nn['analysis'][t])
    arr_qpens = np.asarray(qpens['analysis'][t])
    nn_ens = {v: arr_nn[sl, :].T    for v, sl in slices.items()}   
    q_ens  = {v: arr_qpens[sl, :].T for v, sl in slices.items()}
    return nn_ens, q_ens

rf_models   = {}
max_samples = 200_000

for v in slices:
    X_list, y_list = [], []
    for seed in rf_train_seeds:
        for t in range(T):
            try:
                nn_ens, q_ens = load_ensemble_ncp(seed, t)
            except Exception:
                continue
            nn_v = nn_ens[v]
            q_v  = q_ens[v]
            feat = np.stack([nn_v, q_v], axis=-1)
            X_list.append(feat.reshape(-1, 2))
            y_list.append(np.abs(q_v - nn_v).reshape(-1))

    if not X_list:
        rf_models[v] = None
        continue

    X_all = np.concatenate(X_list)
    y_all = np.concatenate(y_list)

    if len(y_all) > max_samples:
        rng = np.random.default_rng(exp_seed + 123)
        idx = rng.choice(len(y_all), size=max_samples, replace=False)
        X_all, y_all = X_all[idx], y_all[idx]

    rf = RandomForestRegressor(n_estimators=100, max_depth=20,
                               random_state=exp_seed, n_jobs=1)
    rf.fit(X_all, y_all)
    rf_models[v] = rf
    print(f"RF ready for {v}: n={len(y_all)}, "
          f"MAE={mean_absolute_error(y_all, rf.predict(X_all)):.6f}")
    
coverage_all_ncp      = {v: np.zeros((I, T)) for v in slices}
interval_size_all_ncp = {v: np.zeros((I, T)) for v in slices}

for i in range(I):
    print(f"\n=== Iteration {i+1}/{I} ===")
    seeds = all_eval_seeds.copy()
    random.shuffle(seeds)
    calib_seeds = seeds[:len(seeds)//2]
    test_seeds  = seeds[len(seeds)//2:]
    print(f"calib={calib_seeds}, test={test_seeds}")

    for t in range(T_start, T): 
        nn_cal, qp_cal = {v: [] for v in slices}, {v: [] for v in slices}
        for seed in calib_seeds:
            try:
                nn_ens, q_ens = load_ensemble_ncp(seed, t)
                for v in slices:
                    nn_cal[v].append(nn_ens[v])
                    qp_cal[v].append(q_ens[v])
            except Exception as e:
                print(f"calib seed={seed}, t={t}: {e}")

        thresh = {}
        for v in slices:
            if not nn_cal[v]:
                thresh[v] = np.nan
                continue
            nn_stack = np.stack(nn_cal[v], axis=0)   
            qp_stack = np.stack(qp_cal[v], axis=0)
            res = np.abs(qp_stack - nn_stack)
            if v in ('u', 'h') and rf_models.get(v) is not None:
                feat  = np.stack([nn_stack, qp_stack], axis=-1)
                sigma = rf_models[v].predict(feat.reshape(-1, 2)).reshape(nn_stack.shape)
                scores = res / sigma
            else:
                scores = res
            scores = scores[np.isfinite(scores)]
            N      = scores.size
            qprob  = float(np.ceil((N + 1) * (1 - alpha)) / N)
            thresh[v] = float(np.quantile(scores.ravel(), qprob, method="inverted_cdf"))

        nn_test, qp_test = {v: [] for v in slices}, {v: [] for v in slices}
        for seed in test_seeds:
            try:
                nn_ens, q_ens = load_ensemble_ncp(seed, t)
                for v in slices:
                    nn_test[v].append(nn_ens[v])
                    qp_test[v].append(q_ens[v])
            except Exception as e:
                print(f"test seed={seed}, t={t}: {e}")

        for v in slices:
            qv = thresh[v]
            if not np.isfinite(qv) or not nn_test[v]:
                coverage_all_ncp[v][i, t] = np.nan
                interval_size_all_ncp[v][i, t] = np.nan
                continue
            nn_stack = np.stack(nn_test[v], axis=0)
            qp_stack = np.stack(qp_test[v], axis=0)
            if v in ('u', 'h') and rf_models.get(v) is not None:
                feat  = np.stack([nn_stack, qp_stack], axis=-1)
                sigma = rf_models[v].predict(feat.reshape(-1, 2)).reshape(nn_stack.shape)
                band  = qv * sigma
            else:
                band = qv
            lower = np.maximum(nn_stack - band, 0)
            upper = np.maximum(nn_stack + band, 0)
            in_int = (qp_stack >= lower) & (qp_stack <= upper)
            coverage_all_ncp[v][i, t]      = 100.0 * float(np.mean(in_int))
            interval_size_all_ncp[v][i, t] = float(np.mean(upper - lower))

        print(f"  t={t}: " + "  ".join(
            f"{v} q={thresh[v]:.4f} cov={coverage_all_ncp[v][i,t]:.1f}%"
            for v in slices))
        
export_ncp = {'time_step': np.arange(T_start, T)}
for v in slices:
    export_ncp[f'{v}_avg_coverage']      = np.nanmean(coverage_all_ncp[v],      axis=0)[T_start:]
    export_ncp[f'{v}_std_coverage']      = np.nanstd(coverage_all_ncp[v],       axis=0)[T_start:]
    export_ncp[f'{v}_avg_interval_size'] = np.nanmean(interval_size_all_ncp[v], axis=0)[T_start:]
    export_ncp[f'{v}_std_interval_size'] = np.nanstd(interval_size_all_ncp[v],  axis=0)[T_start:]

df_ncp = pd.DataFrame(export_ncp).set_index('time_step')
csv_path_ncp = "ncp_coverage_interval.csv"
df_ncp.to_csv(csv_path_ncp)
print(f"Saved: {csv_path_ncp}")
print(df_ncp.head())        

RF ready for u: n=200000, MAE=0.000137
RF ready for h: n=200000, MAE=0.000539
RF ready for r: n=200000, MAE=0.000027

=== Iteration 1/10 ===
calib=[62, 60, 69, 64, 63], test=[65, 61, 66, 68, 67]
  t=20: u q=1.3842 cov=90.2%  h q=1.2843 cov=90.6%  r q=0.0057 cov=89.0%
  t=21: u q=1.4068 cov=90.3%  h q=1.3283 cov=90.9%  r q=0.0045 cov=88.6%
  t=22: u q=1.3771 cov=89.0%  h q=1.3398 cov=90.6%  r q=0.0030 cov=88.9%
  t=23: u q=1.4053 cov=91.2%  h q=1.3400 cov=90.6%  r q=0.0023 cov=89.0%
  t=24: u q=1.3524 cov=89.2%  h q=1.3235 cov=90.1%  r q=0.0015 cov=87.4%
  t=25: u q=1.3704 cov=91.3%  h q=1.3246 cov=90.3%  r q=0.0011 cov=86.9%
  t=26: u q=1.3487 cov=89.0%  h q=1.3070 cov=89.2%  r q=0.0013 cov=86.8%
  t=27: u q=1.3877 cov=89.7%  h q=1.3402 cov=90.0%  r q=0.0011 cov=87.1%
  t=28: u q=1.3494 cov=89.6%  h q=1.3475 cov=91.0%  r q=0.0012 cov=88.5%
  t=29: u q=1.3467 cov=89.6%  h q=1.3091 cov=89.0%  r q=0.0011 cov=88.3%
  t=30: u q=1.3688 cov=89.8%  h q=1.3179 cov=89.3%  r q=0.0013 cov=88.5%
  

  t=130: u q=1.3420 cov=90.4%  h q=1.3176 cov=90.4%  r q=0.0009 cov=90.2%
  t=131: u q=1.3207 cov=89.3%  h q=1.3050 cov=89.8%  r q=0.0009 cov=89.7%
  t=132: u q=1.3050 cov=88.6%  h q=1.3280 cov=90.1%  r q=0.0010 cov=90.9%
  t=133: u q=1.3192 cov=89.6%  h q=1.3224 cov=89.8%  r q=0.0012 cov=92.0%
  t=134: u q=1.3136 cov=88.9%  h q=1.3151 cov=89.9%  r q=0.0013 cov=92.2%
  t=135: u q=1.3147 cov=89.8%  h q=1.3284 cov=89.9%  r q=0.0011 cov=91.0%
  t=136: u q=1.3392 cov=90.7%  h q=1.3021 cov=89.6%  r q=0.0008 cov=91.4%
  t=137: u q=1.3120 cov=89.6%  h q=1.3124 cov=89.5%  r q=0.0009 cov=92.1%
  t=138: u q=1.3144 cov=88.7%  h q=1.3295 cov=90.6%  r q=0.0009 cov=90.6%
  t=139: u q=1.3213 cov=89.4%  h q=1.3167 cov=90.1%  r q=0.0011 cov=91.0%
  t=140: u q=1.3064 cov=89.8%  h q=1.3255 cov=90.3%  r q=0.0016 cov=92.5%
  t=141: u q=1.3039 cov=89.2%  h q=1.3182 cov=90.5%  r q=0.0014 cov=90.0%
  t=142: u q=1.3303 cov=89.9%  h q=1.3035 cov=89.6%  r q=0.0012 cov=91.3%
  t=143: u q=1.2994 cov=89.4%  h q=1.3

  t=60: u q=1.3945 cov=92.0%  h q=1.3424 cov=91.1%  r q=0.0015 cov=91.3%
  t=61: u q=1.3661 cov=91.1%  h q=1.3215 cov=90.3%  r q=0.0011 cov=88.1%
  t=62: u q=1.3674 cov=90.0%  h q=1.3352 cov=91.1%  r q=0.0007 cov=85.0%
  t=63: u q=1.3617 cov=90.6%  h q=1.2907 cov=88.8%  r q=0.0016 cov=90.1%
  t=64: u q=1.3094 cov=89.0%  h q=1.3070 cov=88.8%  r q=0.0013 cov=89.9%
  t=65: u q=1.3304 cov=89.5%  h q=1.3289 cov=90.5%  r q=0.0015 cov=91.6%
  t=66: u q=1.3503 cov=90.4%  h q=1.3268 cov=90.0%  r q=0.0011 cov=88.3%
  t=67: u q=1.3452 cov=89.9%  h q=1.3077 cov=89.7%  r q=0.0009 cov=85.9%
  t=68: u q=1.3506 cov=90.4%  h q=1.3018 cov=89.1%  r q=0.0011 cov=87.1%
  t=69: u q=1.3761 cov=91.2%  h q=1.3111 cov=88.7%  r q=0.0015 cov=89.9%
  t=70: u q=1.3537 cov=91.2%  h q=1.3002 cov=89.5%  r q=0.0016 cov=90.6%
  t=71: u q=1.3022 cov=89.6%  h q=1.3083 cov=89.2%  r q=0.0013 cov=88.5%
  t=72: u q=1.3081 cov=89.8%  h q=1.3329 cov=91.0%  r q=0.0015 cov=90.0%
  t=73: u q=1.3358 cov=88.9%  h q=1.3019 cov=89.5% 

  t=172: u q=1.3170 cov=90.1%  h q=1.3562 cov=90.6%  r q=0.0011 cov=91.2%
  t=173: u q=1.3146 cov=89.7%  h q=1.3448 cov=90.4%  r q=0.0011 cov=90.4%
  t=174: u q=1.3547 cov=90.4%  h q=1.3353 cov=90.9%  r q=0.0012 cov=91.2%
  t=175: u q=1.3096 cov=89.8%  h q=1.3204 cov=90.1%  r q=0.0014 cov=91.1%
  t=176: u q=1.3667 cov=89.6%  h q=1.3156 cov=90.2%  r q=0.0020 cov=93.1%
  t=177: u q=1.3639 cov=90.7%  h q=1.3408 cov=90.6%  r q=0.0014 cov=90.9%
  t=178: u q=1.3725 cov=90.5%  h q=1.3305 cov=89.8%  r q=0.0013 cov=92.7%
  t=179: u q=1.3427 cov=89.6%  h q=1.3189 cov=89.8%  r q=0.0013 cov=94.2%
  t=180: u q=1.3425 cov=90.5%  h q=1.3272 cov=90.5%  r q=0.0012 cov=93.1%
  t=181: u q=1.3749 cov=91.5%  h q=1.3280 cov=90.6%  r q=0.0010 cov=92.9%
  t=182: u q=1.3551 cov=91.6%  h q=1.3424 cov=90.9%  r q=0.0013 cov=94.2%
  t=183: u q=1.3360 cov=91.6%  h q=1.3362 cov=90.9%  r q=0.0013 cov=93.1%
  t=184: u q=1.3034 cov=90.2%  h q=1.3096 cov=90.1%  r q=0.0017 cov=93.5%
  t=185: u q=1.3491 cov=90.4%  h q=1.3

  t=102: u q=1.2887 cov=88.6%  h q=1.3099 cov=90.0%  r q=0.0013 cov=91.3%
  t=103: u q=1.2653 cov=88.1%  h q=1.3001 cov=89.3%  r q=0.0011 cov=88.7%
  t=104: u q=1.3061 cov=89.2%  h q=1.2903 cov=88.4%  r q=0.0015 cov=90.4%
  t=105: u q=1.3018 cov=90.2%  h q=1.3257 cov=89.5%  r q=0.0017 cov=91.3%
  t=106: u q=1.3116 cov=90.0%  h q=1.3103 cov=90.3%  r q=0.0020 cov=92.7%
  t=107: u q=1.3486 cov=89.9%  h q=1.3286 cov=90.4%  r q=0.0019 cov=91.8%
  t=108: u q=1.3317 cov=90.2%  h q=1.3289 cov=89.9%  r q=0.0016 cov=90.7%
  t=109: u q=1.3550 cov=90.5%  h q=1.3248 cov=90.1%  r q=0.0009 cov=87.5%
  t=110: u q=1.3315 cov=90.4%  h q=1.3210 cov=89.4%  r q=0.0008 cov=88.9%
  t=111: u q=1.3626 cov=90.5%  h q=1.3275 cov=89.9%  r q=0.0007 cov=88.2%
  t=112: u q=1.3394 cov=91.1%  h q=1.3306 cov=90.2%  r q=0.0009 cov=89.2%
  t=113: u q=1.3797 cov=91.5%  h q=1.3070 cov=89.9%  r q=0.0010 cov=90.5%
  t=114: u q=1.3042 cov=89.8%  h q=1.3042 cov=88.7%  r q=0.0014 cov=92.2%
  t=115: u q=1.3100 cov=89.9%  h q=1.3

...

  t=86: u q=1.3290 cov=89.8%  h q=1.3176 cov=89.6%  r q=0.0012 cov=89.4%
  t=87: u q=1.3436 cov=90.0%  h q=1.2981 cov=90.1%  r q=0.0011 cov=87.9%
  t=88: u q=1.3549 cov=89.5%  h q=1.3402 cov=91.0%  r q=0.0013 cov=87.4%
  t=89: u q=1.3396 cov=89.9%  h q=1.3621 cov=91.3%  r q=0.0013 cov=87.1%
  t=90: u q=1.3052 cov=89.1%  h q=1.3129 cov=90.1%  r q=0.0013 cov=86.2%
  t=91: u q=1.3508 cov=90.6%  h q=1.3440 cov=90.7%  r q=0.0013 cov=88.0%
  t=92: u q=1.3113 cov=89.4%  h q=1.3375 cov=89.2%  r q=0.0012 cov=87.9%
  t=93: u q=1.3282 cov=89.6%  h q=1.2970 cov=89.4%  r q=0.0012 cov=89.2%
  t=94: u q=1.3370 cov=89.6%  h q=1.3124 cov=89.9%  r q=0.0016 cov=90.4%
  t=95: u q=1.3164 cov=89.0%  h q=1.3118 cov=89.0%  r q=0.0010 cov=90.3%
  t=96: u q=1.3143 cov=89.0%  h q=1.3179 cov=89.6%  r q=0.0015 cov=92.0%
  t=97: u q=1.3446 cov=89.9%  h q=1.2861 cov=88.8%  r q=0.0011 cov=90.2%
  t=98: u q=1.3619 cov=90.1%  h q=1.3283 cov=89.6%  r q=0.0012 cov=90.3%
  t=99: u q=1.3207 cov=90.4%  h q=1.3291 cov=90.0% 

  t=197: u q=1.3237 cov=90.2%  h q=1.3408 cov=90.1%  r q=0.0018 cov=91.7%
  t=198: u q=1.3671 cov=91.7%  h q=1.3453 cov=91.7%  r q=0.0016 cov=89.9%
  t=199: u q=1.3907 cov=91.9%  h q=1.3379 cov=90.5%  r q=0.0012 cov=89.3%
  t=200: u q=1.3883 cov=91.7%  h q=1.3405 cov=91.3%  r q=0.0010 cov=88.3%

=== Iteration 9/10 ===
calib=[62, 66, 64, 63, 67], test=[68, 60, 61, 69, 65]
  t=20: u q=1.4036 cov=90.9%  h q=1.2913 cov=91.0%  r q=0.0060 cov=90.3%
  t=21: u q=1.3791 cov=89.0%  h q=1.3236 cov=90.7%  r q=0.0049 cov=90.3%
  t=22: u q=1.3996 cov=89.9%  h q=1.3286 cov=90.1%  r q=0.0033 cov=89.7%
  t=23: u q=1.3982 cov=90.9%  h q=1.3315 cov=90.2%  r q=0.0023 cov=89.1%
  t=24: u q=1.3707 cov=90.0%  h q=1.3120 cov=89.5%  r q=0.0018 cov=89.3%
  t=25: u q=1.3528 cov=90.5%  h q=1.3360 cov=91.0%  r q=0.0015 cov=89.7%
  t=26: u q=1.3703 cov=89.9%  h q=1.3107 cov=89.4%  r q=0.0014 cov=87.7%
  t=27: u q=1.3872 cov=89.7%  h q=1.3127 cov=88.3%  r q=0.0013 cov=88.8%
  t=28: u q=1.3591 cov=90.1%  h q=1.3232 c

  t=127: u q=1.3812 cov=90.0%  h q=1.3144 cov=89.4%  r q=0.0012 cov=91.3%
  t=128: u q=1.3504 cov=89.4%  h q=1.3195 cov=89.0%  r q=0.0017 cov=92.6%
  t=129: u q=1.3578 cov=89.6%  h q=1.3374 cov=90.6%  r q=0.0016 cov=93.0%
  t=130: u q=1.3336 cov=89.9%  h q=1.3277 cov=91.0%  r q=0.0013 cov=94.3%
  t=131: u q=1.3597 cov=91.2%  h q=1.3054 cov=89.9%  r q=0.0015 cov=94.4%
  t=132: u q=1.3386 cov=90.2%  h q=1.3340 cov=90.4%  r q=0.0013 cov=93.2%
  t=133: u q=1.3041 cov=88.9%  h q=1.3282 cov=90.2%  r q=0.0012 cov=92.1%
  t=134: u q=1.3418 cov=90.2%  h q=1.3176 cov=90.1%  r q=0.0012 cov=92.0%
  t=135: u q=1.3062 cov=89.4%  h q=1.3398 cov=90.5%  r q=0.0011 cov=90.7%
  t=136: u q=1.3469 cov=91.0%  h q=1.2824 cov=88.1%  r q=0.0006 cov=88.7%
  t=137: u q=1.3024 cov=89.3%  h q=1.3124 cov=89.5%  r q=0.0007 cov=89.7%
  t=138: u q=1.3637 cov=90.9%  h q=1.3047 cov=89.1%  r q=0.0008 cov=90.0%
  t=139: u q=1.3413 cov=90.3%  h q=1.3220 cov=90.5%  r q=0.0009 cov=89.8%
  t=140: u q=1.3100 cov=90.0%  h q=1.3

  t=57: u q=1.3388 cov=90.2%  h q=1.3248 cov=89.8%  r q=0.0020 cov=93.2%
  t=58: u q=1.3423 cov=89.5%  h q=1.3068 cov=89.5%  r q=0.0020 cov=93.2%
  t=59: u q=1.3578 cov=90.5%  h q=1.2918 cov=90.2%  r q=0.0025 cov=93.8%
  t=60: u q=1.3222 cov=88.9%  h q=1.3147 cov=89.5%  r q=0.0017 cov=92.7%
  t=61: u q=1.3228 cov=89.1%  h q=1.3017 cov=88.9%  r q=0.0018 cov=92.8%
  t=62: u q=1.3581 cov=89.7%  h q=1.3188 cov=90.1%  r q=0.0019 cov=94.5%
  t=63: u q=1.3589 cov=90.5%  h q=1.3112 cov=90.1%  r q=0.0022 cov=93.7%
  t=64: u q=1.3442 cov=90.5%  h q=1.3217 cov=89.7%  r q=0.0017 cov=92.8%
  t=65: u q=1.3604 cov=90.8%  h q=1.3202 cov=89.9%  r q=0.0015 cov=92.0%
  t=66: u q=1.3466 cov=90.2%  h q=1.3122 cov=89.0%  r q=0.0017 cov=92.4%
  t=67: u q=1.3310 cov=89.1%  h q=1.3225 cov=90.7%  r q=0.0016 cov=91.5%
  t=68: u q=1.3241 cov=89.3%  h q=1.3242 cov=90.6%  r q=0.0018 cov=91.9%
  t=69: u q=1.3146 cov=88.5%  h q=1.3239 cov=89.4%  r q=0.0017 cov=90.9%
  t=70: u q=1.3242 cov=89.9%  h q=1.2990 cov=89.5% 

  t=169: u q=1.3423 cov=89.7%  h q=1.3137 cov=89.6%  r q=0.0010 cov=89.6%
  t=170: u q=1.3270 cov=90.0%  h q=1.3488 cov=90.7%  r q=0.0011 cov=88.9%
  t=171: u q=1.3197 cov=89.6%  h q=1.3288 cov=89.4%  r q=0.0008 cov=86.8%
  t=172: u q=1.3112 cov=89.9%  h q=1.3449 cov=89.9%  r q=0.0007 cov=87.8%
  t=173: u q=1.2984 cov=89.0%  h q=1.3436 cov=90.3%  r q=0.0007 cov=86.3%
  t=174: u q=1.3283 cov=89.3%  h q=1.3032 cov=88.9%  r q=0.0007 cov=86.3%
  t=175: u q=1.3053 cov=89.5%  h q=1.3050 cov=89.3%  r q=0.0009 cov=86.9%
  t=176: u q=1.3529 cov=89.0%  h q=1.2972 cov=89.1%  r q=0.0012 cov=87.6%
  t=177: u q=1.3508 cov=90.2%  h q=1.3168 cov=89.0%  r q=0.0009 cov=87.3%
  t=178: u q=1.3446 cov=89.2%  h q=1.3325 cov=90.0%  r q=0.0008 cov=88.7%
  t=179: u q=1.3464 cov=89.8%  h q=1.3164 cov=89.5%  r q=0.0007 cov=88.3%
  t=180: u q=1.3011 cov=88.8%  h q=1.3117 cov=89.4%  r q=0.0007 cov=88.9%
  t=181: u q=1.3310 cov=89.7%  h q=1.3073 cov=89.3%  r q=0.0008 cov=90.8%
  t=182: u q=1.3192 cov=90.1%  h q=1.3

# CQR Method - Coverage and Interval Size

In [23]:
base_path      = r"C:\Users\MGA5500\Desktop\PROJECT\WW\MSW codes Reimp"
all_eval_seeds = list(range(60, 70))
nx             = 250
alpha          = 0.10
slices         = {'u': slice(0, nx), 'h': slice(nx, 2*nx), 'r': slice(2*nx, 3*nx)}
VARS           = ['u', 'h', 'r']
I              = 10
T_start        = 20
T              = 201

In [24]:
def qhat(scores, a):
    s = np.asarray(scores).ravel()
    n = s.size
    if n == 0:
        return 0.0
    qprob = float(np.ceil((n + 1) * (1 - a)) / n)
    return float(np.quantile(s, min(qprob, 1.0), method="inverted_cdf"))

coverage_all_cqr      = {v: np.zeros((I, T)) for v in VARS}
interval_size_all_cqr = {v: np.zeros((I, T)) for v in VARS}

for i in range(I):
    print(f"\n=== CQR Iteration {i+1}/{I} ===")
    seeds = all_eval_seeds.copy()
    np.random.shuffle(seeds)
    calib_seeds = seeds[:len(seeds)//2]
    test_seeds  = seeds[len(seeds)//2:]
    print(f"calib={calib_seeds}, test={test_seeds}")

    for t in range(T_start, T):
        scores_cal = {v: [] for v in VARS}
        for seed in calib_seeds:
            path = os.path.join(base_path, str(seed))
            with open(os.path.join(path, 'QPEns_data.pkl'), 'rb') as f:
                qpens = pickle.load(f)
            with open(os.path.join(path, 'NN_data.pkl'), 'rb') as f:
                nn = pickle.load(f)
            q_arr = np.asarray(qpens['analysis'][t])          
            lo    = np.asarray(nn['lower'][t])
            hi    = np.asarray(nn['upper'][t])   
            for v in VARS:
                sl = slices[v]
                s  = np.maximum(np.maximum(lo[sl, :] - q_arr[sl, :],
                                           q_arr[sl, :] - hi[sl, :]), 0.0)
                scores_cal[v].append(s.ravel())

        deltas = {v: qhat(np.concatenate(scores_cal[v]), alpha) if scores_cal[v] else 0.0
                  for v in VARS}

        cov_vals = {v: [] for v in VARS}
        w_vals   = {v: [] for v in VARS}
        for seed in test_seeds:
            path = os.path.join(base_path, str(seed))
            with open(os.path.join(path, 'QPEns_data.pkl'), 'rb') as f:
                qpens = pickle.load(f)
            with open(os.path.join(path, 'NN_data.pkl'), 'rb') as f:
                nn = pickle.load(f)

            q_arr = np.asarray(qpens['analysis'][t])          
            lo    = np.asarray(nn['lower'][t])
            hi    = np.asarray(nn['upper'][t]) 

            for v in VARS:
                sl      = slices[v]
                lo[sl]  = np.maximum(lo[sl] - deltas[v], 0.0)
                hi[sl]  = np.maximum(hi[sl] + deltas[v], 0.0)

            for v in VARS:
                sl     = slices[v]
                inside = (q_arr[sl, :] >= lo[sl, :]) & (q_arr[sl, :] <= hi[sl, :])
                cov_vals[v].append(float(np.mean(inside)))
                w_vals[v].append(float(np.mean(hi[sl] - lo[sl])))

        for v in VARS:
            coverage_all_cqr[v][i, t]      = 100.0 * np.mean(cov_vals[v]) if cov_vals[v] else np.nan
            interval_size_all_cqr[v][i, t] = np.mean(w_vals[v])           if w_vals[v]   else np.nan

        print(f"  t={t}: " + "  ".join(
            f"{v} d={deltas[v]:.4f} cov={coverage_all_cqr[v][i,t]:.1f}%"
            for v in VARS))

# Just change the export line
export_cqr = {'time_step': np.arange(T_start, T)}
for v in VARS:
    export_cqr[f'{v}_avg_coverage']      = np.nanmean(coverage_all_cqr[v], axis=0)[T_start:]
    export_cqr[f'{v}_std_coverage']      = np.nanstd(coverage_all_cqr[v],  axis=0)[T_start:]
    export_cqr[f'{v}_avg_interval_size'] = np.nanmean(interval_size_all_cqr[v], axis=0)[T_start:]
    export_cqr[f'{v}_std_interval_size'] = np.nanstd(interval_size_all_cqr[v],  axis=0)[T_start:]

df_cqr = pd.DataFrame(export_cqr).set_index('time_step')
csv_path_cqr = "cqr_coverage_interval.csv"
df_cqr.to_csv(csv_path_cqr)
print(f"Saved: {csv_path_cqr}")
print(df_cqr.head())


=== CQR Iteration 1/10 ===
calib=[63, 68, 61, 62, 64], test=[69, 65, 67, 60, 66]
  t=20: u d=0.0024 cov=87.8%  h d=0.0499 cov=90.3%  r d=0.0029 cov=90.4%
  t=21: u d=0.0025 cov=90.1%  h d=0.0385 cov=91.2%  r d=0.0021 cov=90.1%
  t=22: u d=0.0024 cov=91.5%  h d=0.0232 cov=91.5%  r d=0.0013 cov=91.0%
  t=23: u d=0.0020 cov=90.6%  h d=0.0132 cov=90.6%  r d=0.0008 cov=90.8%
  t=24: u d=0.0019 cov=91.0%  h d=0.0089 cov=90.9%  r d=0.0003 cov=90.0%
  t=25: u d=0.0019 cov=91.0%  h d=0.0087 cov=89.8%  r d=0.0003 cov=90.2%
  t=26: u d=0.0019 cov=89.2%  h d=0.0080 cov=87.8%  r d=0.0003 cov=86.9%
  t=27: u d=0.0017 cov=87.6%  h d=0.0064 cov=85.5%  r d=0.0001 cov=85.0%
  t=28: u d=0.0017 cov=87.0%  h d=0.0071 cov=87.3%  r d=0.0001 cov=85.0%
  t=29: u d=0.0016 cov=86.9%  h d=0.0067 cov=87.3%  r d=0.0001 cov=85.3%
  t=30: u d=0.0019 cov=90.3%  h d=0.0095 cov=89.2%  r d=0.0004 cov=88.7%
  t=31: u d=0.0019 cov=91.0%  h d=0.0111 cov=92.2%  r d=0.0004 cov=91.1%
  t=32: u d=0.0020 cov=92.4%  h d=0.0095 c

  t=131: u d=0.0015 cov=84.5%  h d=0.0061 cov=86.6%  r d=0.0001 cov=88.0%
  t=132: u d=0.0016 cov=87.3%  h d=0.0064 cov=89.4%  r d=0.0002 cov=90.7%
  t=133: u d=0.0016 cov=88.7%  h d=0.0074 cov=91.4%  r d=0.0004 cov=91.2%
  t=134: u d=0.0016 cov=89.0%  h d=0.0078 cov=91.3%  r d=0.0001 cov=88.8%
  t=135: u d=0.0015 cov=88.1%  h d=0.0062 cov=89.4%  r d=0.0002 cov=89.7%
  t=136: u d=0.0015 cov=88.9%  h d=0.0053 cov=89.3%  r d=0.0001 cov=88.3%
  t=137: u d=0.0015 cov=88.6%  h d=0.0057 cov=88.5%  r d=0.0001 cov=89.1%
  t=138: u d=0.0016 cov=89.6%  h d=0.0061 cov=88.0%  r d=0.0001 cov=88.8%
  t=139: u d=0.0018 cov=91.6%  h d=0.0084 cov=91.5%  r d=0.0002 cov=90.1%
  t=140: u d=0.0020 cov=92.8%  h d=0.0110 cov=92.8%  r d=0.0005 cov=91.1%
  t=141: u d=0.0020 cov=92.5%  h d=0.0085 cov=91.2%  r d=0.0004 cov=90.5%
  t=142: u d=0.0020 cov=91.2%  h d=0.0087 cov=91.0%  r d=0.0003 cov=90.0%
  t=143: u d=0.0018 cov=90.2%  h d=0.0079 cov=89.8%  r d=0.0004 cov=90.5%
  t=144: u d=0.0021 cov=92.1%  h d=0.0

  t=61: u d=0.0016 cov=89.3%  h d=0.0080 cov=90.7%  r d=0.0003 cov=90.6%
  t=62: u d=0.0018 cov=90.9%  h d=0.0097 cov=93.0%  r d=0.0006 cov=93.2%
  t=63: u d=0.0020 cov=92.5%  h d=0.0113 cov=93.5%  r d=0.0007 cov=93.2%
  t=64: u d=0.0019 cov=92.0%  h d=0.0087 cov=92.1%  r d=0.0006 cov=92.8%
  t=65: u d=0.0019 cov=92.7%  h d=0.0095 cov=91.7%  r d=0.0005 cov=91.1%
  t=66: u d=0.0020 cov=93.7%  h d=0.0093 cov=92.8%  r d=0.0006 cov=92.1%
  t=67: u d=0.0021 cov=92.2%  h d=0.0114 cov=93.2%  r d=0.0006 cov=92.9%
  t=68: u d=0.0019 cov=90.4%  h d=0.0113 cov=91.0%  r d=0.0005 cov=89.9%
  t=69: u d=0.0018 cov=89.9%  h d=0.0080 cov=89.7%  r d=0.0004 cov=89.5%
  t=70: u d=0.0017 cov=88.1%  h d=0.0080 cov=89.3%  r d=0.0004 cov=88.6%
  t=71: u d=0.0017 cov=87.7%  h d=0.0083 cov=88.8%  r d=0.0003 cov=88.9%
  t=72: u d=0.0019 cov=90.4%  h d=0.0080 cov=89.6%  r d=0.0003 cov=88.8%
  t=73: u d=0.0018 cov=89.8%  h d=0.0085 cov=89.6%  r d=0.0003 cov=90.1%
  t=74: u d=0.0017 cov=89.8%  h d=0.0067 cov=89.5% 

  t=173: u d=0.0015 cov=87.5%  h d=0.0058 cov=87.6%  r d=0.0002 cov=89.8%
  t=174: u d=0.0016 cov=88.9%  h d=0.0065 cov=89.2%  r d=0.0001 cov=88.8%
  t=175: u d=0.0016 cov=86.4%  h d=0.0065 cov=88.0%  r d=0.0002 cov=88.5%
  t=176: u d=0.0018 cov=89.8%  h d=0.0076 cov=87.9%  r d=0.0001 cov=84.9%
  t=177: u d=0.0018 cov=88.9%  h d=0.0073 cov=87.4%  r d=0.0003 cov=88.1%
  t=178: u d=0.0016 cov=87.4%  h d=0.0057 cov=86.3%  r d=0.0000 cov=84.9%
  t=179: u d=0.0016 cov=88.4%  h d=0.0057 cov=87.0%  r d=0.0000 cov=84.8%
  t=180: u d=0.0015 cov=87.4%  h d=0.0051 cov=85.9%  r d=0.0000 cov=85.8%
  t=181: u d=0.0014 cov=87.0%  h d=0.0049 cov=84.5%  r d=0.0000 cov=85.9%
  t=182: u d=0.0016 cov=88.1%  h d=0.0060 cov=87.5%  r d=0.0001 cov=87.2%
  t=183: u d=0.0015 cov=87.8%  h d=0.0058 cov=86.5%  r d=0.0000 cov=84.9%
  t=184: u d=0.0017 cov=89.3%  h d=0.0061 cov=86.5%  r d=0.0001 cov=87.2%
  t=185: u d=0.0019 cov=90.0%  h d=0.0099 cov=90.0%  r d=0.0003 cov=88.8%
  t=186: u d=0.0018 cov=89.0%  h d=0.0

  t=103: u d=0.0016 cov=87.5%  h d=0.0090 cov=89.3%  r d=0.0004 cov=90.2%
  t=104: u d=0.0016 cov=86.9%  h d=0.0101 cov=90.5%  r d=0.0004 cov=89.8%
  t=105: u d=0.0019 cov=89.3%  h d=0.0113 cov=91.1%  r d=0.0006 cov=91.3%
  t=106: u d=0.0019 cov=90.8%  h d=0.0093 cov=90.3%  r d=0.0004 cov=90.2%
  t=107: u d=0.0018 cov=89.8%  h d=0.0076 cov=89.1%  r d=0.0004 cov=90.9%
  t=108: u d=0.0018 cov=88.9%  h d=0.0080 cov=90.3%  r d=0.0002 cov=87.9%
  t=109: u d=0.0016 cov=88.2%  h d=0.0069 cov=89.2%  r d=0.0002 cov=88.7%
  t=110: u d=0.0016 cov=90.0%  h d=0.0061 cov=89.7%  r d=0.0001 cov=85.5%
  t=111: u d=0.0016 cov=89.8%  h d=0.0061 cov=88.7%  r d=0.0000 cov=86.4%
  t=112: u d=0.0016 cov=90.1%  h d=0.0063 cov=89.6%  r d=0.0000 cov=87.1%
  t=113: u d=0.0017 cov=89.7%  h d=0.0071 cov=89.5%  r d=0.0002 cov=87.7%
  t=114: u d=0.0020 cov=91.6%  h d=0.0085 cov=90.4%  r d=0.0003 cov=89.3%
  t=115: u d=0.0018 cov=88.6%  h d=0.0073 cov=88.3%  r d=0.0002 cov=87.5%
  t=116: u d=0.0018 cov=88.9%  h d=0.0

...

  t=87: u d=0.0019 cov=90.6%  h d=0.0094 cov=90.9%  r d=0.0004 cov=89.6%
  t=88: u d=0.0020 cov=90.6%  h d=0.0109 cov=91.0%  r d=0.0007 cov=90.9%
  t=89: u d=0.0018 cov=90.3%  h d=0.0100 cov=91.0%  r d=0.0005 cov=91.0%
  t=90: u d=0.0019 cov=91.1%  h d=0.0117 cov=91.9%  r d=0.0006 cov=90.9%
  t=91: u d=0.0017 cov=89.3%  h d=0.0081 cov=89.3%  r d=0.0003 cov=88.3%
  t=92: u d=0.0019 cov=90.8%  h d=0.0078 cov=89.0%  r d=0.0002 cov=86.9%
  t=93: u d=0.0017 cov=88.5%  h d=0.0075 cov=87.6%  r d=0.0001 cov=86.1%
  t=94: u d=0.0017 cov=88.7%  h d=0.0077 cov=87.7%  r d=0.0002 cov=86.2%
  t=95: u d=0.0017 cov=88.5%  h d=0.0066 cov=88.0%  r d=0.0002 cov=88.3%
  t=96: u d=0.0016 cov=87.8%  h d=0.0079 cov=88.1%  r d=0.0002 cov=87.7%
  t=97: u d=0.0017 cov=88.9%  h d=0.0070 cov=89.3%  r d=0.0001 cov=87.4%
  t=98: u d=0.0016 cov=87.2%  h d=0.0070 cov=88.7%  r d=0.0002 cov=88.5%
  t=99: u d=0.0016 cov=87.0%  h d=0.0056 cov=85.1%  r d=0.0000 cov=84.0%
  t=100: u d=0.0016 cov=88.3%  h d=0.0054 cov=85.6%

  t=198: u d=0.0016 cov=87.1%  h d=0.0069 cov=88.3%  r d=0.0002 cov=87.3%
  t=199: u d=0.0016 cov=87.9%  h d=0.0068 cov=87.8%  r d=0.0002 cov=87.5%
  t=200: u d=0.0018 cov=91.1%  h d=0.0101 cov=92.0%  r d=0.0005 cov=91.3%

=== CQR Iteration 9/10 ===
calib=[67, 66, 62, 65, 60], test=[68, 61, 63, 64, 69]
  t=20: u d=0.0027 cov=92.2%  h d=0.0592 cov=94.1%  r d=0.0033 cov=92.3%
  t=21: u d=0.0027 cov=91.6%  h d=0.0448 cov=93.5%  r d=0.0026 cov=92.7%
  t=22: u d=0.0023 cov=91.1%  h d=0.0260 cov=92.7%  r d=0.0017 cov=92.8%
  t=23: u d=0.0020 cov=90.5%  h d=0.0176 cov=93.0%  r d=0.0010 cov=92.5%
  t=24: u d=0.0019 cov=90.9%  h d=0.0093 cov=91.4%  r d=0.0004 cov=91.7%
  t=25: u d=0.0019 cov=90.2%  h d=0.0097 cov=91.2%  r d=0.0004 cov=91.2%
  t=26: u d=0.0019 cov=89.1%  h d=0.0101 cov=90.9%  r d=0.0007 cov=91.4%
  t=27: u d=0.0020 cov=91.4%  h d=0.0101 cov=92.7%  r d=0.0005 cov=93.2%
  t=28: u d=0.0020 cov=92.2%  h d=0.0102 cov=92.6%  r d=0.0007 cov=93.7%
  t=29: u d=0.0019 cov=92.3%  h d=0.009

  t=128: u d=0.0020 cov=92.5%  h d=0.0118 cov=93.6%  r d=0.0006 cov=93.7%
  t=129: u d=0.0019 cov=92.6%  h d=0.0088 cov=93.2%  r d=0.0004 cov=93.7%
  t=130: u d=0.0018 cov=92.8%  h d=0.0089 cov=94.7%  r d=0.0005 cov=94.5%
  t=131: u d=0.0020 cov=93.9%  h d=0.0101 cov=94.5%  r d=0.0005 cov=93.8%
  t=132: u d=0.0019 cov=93.0%  h d=0.0084 cov=93.6%  r d=0.0003 cov=92.5%
  t=133: u d=0.0017 cov=90.8%  h d=0.0081 cov=92.6%  r d=0.0004 cov=90.6%
  t=134: u d=0.0018 cov=92.8%  h d=0.0088 cov=92.6%  r d=0.0004 cov=92.7%
  t=135: u d=0.0017 cov=92.4%  h d=0.0076 cov=92.3%  r d=0.0004 cov=91.7%
  t=136: u d=0.0017 cov=91.3%  h d=0.0060 cov=91.2%  r d=0.0001 cov=89.9%
  t=137: u d=0.0017 cov=90.9%  h d=0.0065 cov=90.9%  r d=0.0001 cov=91.6%
  t=138: u d=0.0016 cov=89.7%  h d=0.0067 cov=90.0%  r d=0.0002 cov=90.8%
  t=139: u d=0.0016 cov=87.2%  h d=0.0069 cov=88.1%  r d=0.0002 cov=89.2%
  t=140: u d=0.0017 cov=86.9%  h d=0.0076 cov=88.5%  r d=0.0003 cov=89.6%
  t=141: u d=0.0017 cov=87.4%  h d=0.0

  t=58: u d=0.0018 cov=91.4%  h d=0.0107 cov=91.9%  r d=0.0006 cov=91.6%
  t=59: u d=0.0020 cov=91.5%  h d=0.0140 cov=93.4%  r d=0.0009 cov=92.6%
  t=60: u d=0.0017 cov=90.1%  h d=0.0093 cov=90.4%  r d=0.0005 cov=92.4%
  t=61: u d=0.0017 cov=89.8%  h d=0.0073 cov=89.3%  r d=0.0003 cov=90.1%
  t=62: u d=0.0017 cov=90.0%  h d=0.0078 cov=90.0%  r d=0.0003 cov=90.0%
  t=63: u d=0.0018 cov=90.5%  h d=0.0095 cov=91.7%  r d=0.0005 cov=91.4%
  t=64: u d=0.0019 cov=91.6%  h d=0.0082 cov=91.4%  r d=0.0005 cov=92.2%
  t=65: u d=0.0018 cov=89.8%  h d=0.0087 cov=90.6%  r d=0.0004 cov=89.7%
  t=66: u d=0.0018 cov=90.2%  h d=0.0072 cov=89.4%  r d=0.0004 cov=90.1%
  t=67: u d=0.0018 cov=88.2%  h d=0.0067 cov=85.6%  r d=0.0002 cov=86.4%
  t=68: u d=0.0016 cov=86.3%  h d=0.0066 cov=83.8%  r d=0.0002 cov=84.0%
  t=69: u d=0.0016 cov=86.0%  h d=0.0067 cov=87.3%  r d=0.0003 cov=87.3%
  t=70: u d=0.0016 cov=86.4%  h d=0.0068 cov=86.9%  r d=0.0003 cov=87.0%
  t=71: u d=0.0018 cov=88.9%  h d=0.0085 cov=89.0% 

  t=170: u d=0.0017 cov=89.9%  h d=0.0079 cov=91.1%  r d=0.0003 cov=90.2%
  t=171: u d=0.0017 cov=90.4%  h d=0.0072 cov=90.1%  r d=0.0002 cov=89.1%
  t=172: u d=0.0016 cov=90.7%  h d=0.0058 cov=89.9%  r d=0.0001 cov=88.5%
  t=173: u d=0.0017 cov=89.8%  h d=0.0064 cov=89.4%  r d=0.0001 cov=88.8%
  t=174: u d=0.0017 cov=90.7%  h d=0.0072 cov=91.2%  r d=0.0002 cov=90.1%
  t=175: u d=0.0019 cov=92.4%  h d=0.0085 cov=91.9%  r d=0.0003 cov=91.4%
  t=176: u d=0.0018 cov=90.5%  h d=0.0093 cov=90.9%  r d=0.0006 cov=92.0%
  t=177: u d=0.0020 cov=92.1%  h d=0.0102 cov=92.4%  r d=0.0005 cov=90.7%
  t=178: u d=0.0019 cov=92.2%  h d=0.0085 cov=93.4%  r d=0.0003 cov=92.7%
  t=179: u d=0.0017 cov=90.6%  h d=0.0070 cov=91.6%  r d=0.0001 cov=92.3%
  t=180: u d=0.0017 cov=91.4%  h d=0.0077 cov=93.5%  r d=0.0002 cov=93.4%
  t=181: u d=0.0017 cov=92.6%  h d=0.0075 cov=93.6%  r d=0.0003 cov=93.6%
  t=182: u d=0.0018 cov=92.2%  h d=0.0083 cov=92.8%  r d=0.0004 cov=94.2%
  t=183: u d=0.0017 cov=91.3%  h d=0.0